# Sadness of a Song: Predicting How Sad a Song Sounds


| Team member | Part of the project |
|---|---|
| Yuming Zou| Data cleaning, feature engineering, shared train/test split, clustering |
| *TODO* | Data exploration, baseline, linear models (linear, ridge, lasso, logistic regression) |
| *TODO* | kNN and a small neural network |
| *TODO* | Tree models (decision tree, random forest, gradient boosting) |

## Introduction

Some songs sound happy and some sound sad. Can a computer tell which is which just from how a song sounds?

In this project we try to **predict how sad a song is from its audio features**, using a Spotify dataset.
Spotify gives every song a score called **`valence`**, from 0 to 1. It measures how positive a song sounds.
A low valence means the song sounds sadder. Valence is our **target**.

We look at the problem in three ways:
1. **Regression (main task):** predict the `valence` number directly.
2. **Classification:** call a song "sad" if `valence < 0.3`, and predict that label.
3. **Clustering (extra):** group songs by their audio features (without valence), then check the average valence of each group.

We also want to test some common ideas about sad songs:
- Songs in a **minor key** are sadder.
- Songs with **low energy** are sadder.
- Songs with **high acousticness** are sadder.

### The dataset

- Source: [Spotify Tracks Dataset on Kaggle](https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset) (by maharshipandya)
- 114,000 songs: 114 genres with 1,000 songs each, and 21 columns.
- Each row is one song with its name, artist, genre, popularity, and audio features such as `danceability`, `energy`, `acousticness`, `tempo`, and `key`.

### How we work

- **One cleaned dataset, one split.** Everyone uses the same cleaned data and the same 80/20 train/test split (`random_state=42`), so all models are compared fairly. The test set is kept aside and only used for the final scores.
- **Same metrics for every model.**
  - Regression: RMSE, MAE, and R².
  - Classification: F1, ROC-AUC, and the confusion matrix. We don't rely on accuracy alone, because if sad songs are rare, a model that never says "sad" can still look accurate.
- **Start simple.** Every model is compared against a simple baseline (for example, always predicting the average valence). A model is only useful if it beats that.

### Project roadmap

1. **Data cleaning:** remove missing values, duplicate songs, and tracks that are not music, then make the shared train/test split.
2. **Data exploration (training set only):** look at how valence relates to the other features and check our ideas about sad songs.
3. **Feature engineering:** encode categories, scale the features, and make new features.
4. **Modeling:** baseline, linear models, kNN, neural network, and tree models, for both regression and classification, with tuning.
5. **Clustering:** group songs without using valence and see if the groups differ in sadness.
6. **Results:** compare all models, look at where they go wrong, and answer our questions.

### Known limits

- `valence` comes from **Spotify's own model**, not from people rating songs. So we are really predicting another model's score.
- Low valence can mean **sad or angry**. A loud, angry metal song and a quiet, slow piano song can both have low valence. Low valence together with low energy is closer to what we mean by "sad".
- The dataset has exactly 1,000 songs per genre, so it is **not a random sample of all music**. Small genres are over-represented compared with real listening, which may affect what the models learn.

---

# Part 1: Data Cleaning and Feature Engineering

*By Eddy.* This part turns the raw data into the cleaned dataset and the shared train/test split that the rest of the team uses.

## 1. Setup

We load the libraries we need and set a few shared settings.
`RANDOM_STATE = 42` is used everywhere so the team gets the same results every time.

In [103]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

pd.set_option("display.max_columns", None)

RANDOM_STATE = 42
RAW_PATH = "raw_dataset.csv"      # never edit this file

We keep a small log of how many rows each cleaning step removes.
At the end we print it as a table, so the reader can see every step at once.

In [104]:
cleaning_log = []

def log_step(step, before, after, reason):
    """Record how many rows a cleaning step removed and why."""
    cleaning_log.append({
        "step": step,
        "rows_before": len(before),
        "rows_after": len(after),
        "rows_removed": len(before) - len(after),
        "reason": reason,
    })
    print(f"{step}: {len(before):,} -> {len(after):,} rows ({len(before) - len(after):,} removed)")

## 2. First look at the data

Before changing anything, we look at the raw data: how big it is, what type each column is,
and whether anything looks wrong (missing values, odd values, duplicates).

In [105]:
raw = pd.read_csv(RAW_PATH)
print(raw.shape)
raw.head()

(114000, 21)


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [106]:
# Column types and missing values
summary = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "unique": raw.nunique(),
    "example": raw.iloc[0],
})
print(summary, "\n")

print(f"Number of rows with missing value: {len(raw[raw.isna().any(axis=1)])}")



                    dtype  missing  unique                 example
Unnamed: 0          int64        0  114000                       0
track_id              str        0   89741  5SuOikwiRyPMVoIQDJUgSV
artists               str        1   31437             Gen Hoshino
album_name            str        1   46589                  Comedy
track_name            str        1   73608                  Comedy
popularity          int64        0     101                      73
duration_ms         int64        0   50697                  230666
explicit             bool        0       2                   False
danceability      float64        0    1174                   0.676
energy            float64        0    2083                   0.461
key                 int64        0      12                       1
loudness          float64        0   19480                  -6.746
mode                int64        0       2                       0
speechiness       float64        0    1489                   0

In [107]:
# How many songs appear more than once?
n_ids_repeated = (raw["track_id"].value_counts() > 1).sum()
print("Songs that appear more than once:", n_ids_repeated)

n_dup_id = raw.duplicated("track_id").sum()
print("Extra rows with a repeated track_id:", n_dup_id)


Songs that appear more than once: 16641
Extra rows with a repeated track_id: 24259


In [108]:
# Odd values: things that are not possible for a real song, or that suggest the track is not music
odd = pd.Series({
    "tempo == 0 (no beat found)": (raw["tempo"] == 0).sum(),
    "speechiness > 0.66 (mostly spoken words)": (raw["speechiness"] > 0.66).sum(),
    "time_signature == 0": (raw["time_signature"] == 0).sum(),
    "time_signature == 1": (raw["time_signature"] == 1).sum(),
    "duration_ms == 0": (raw["duration_ms"] == 0).sum(),
    "key == -1 (no key found)": (raw["key"] == -1).sum(),
}, name="number of tracks")
print(odd, "\n")

print("time_signature counts:")
print(raw["time_signature"].value_counts().sort_index())

tempo == 0 (no beat found)                  157
speechiness > 0.66 (mostly spoken words)    880
time_signature == 0                         163
time_signature == 1                         973
duration_ms == 0                              1
key == -1 (no key found)                      0
Name: number of tracks, dtype: int64 

time_signature counts:
time_signature
0       163
1       973
3      9195
4    101843
5      1826
Name: count, dtype: int64


**What we found:**

- **Missing values:** Only 1 row out of 114,000 has missing values. It has no artist, album, or track name. We can simply drop it.
- **Duplicate songs:** This is the biggest problem.
  - 16,641 songs appear more than once with the same `track_id`, because they are listed under several genres. These songs take up 40,900 rows in total. If we keep one row per song, 24,259 rows are extra copies that we will remove.
  - We will keep one record for each song. Otherwise the same song could end up in both the train and test sets, and our test scores would look better than they really are.
- **Odd values:**
  - 157 tracks have `tempo == 0`, which usually means white noise or no beat.
  - 880 tracks have `speechiness > 0.66`. Spotify says these are almost all spoken words (podcasts, audiobooks, comedy), not music.
  - `time_signature` has 163 tracks with 0 and 973 with 1. A song can't have 0 beats per bar, and 1 is very rare, so these are most likely detection errors.
  - 1 track has `duration_ms == 0`. It is the same row that has missing values, so dropping that row fixes this too.
  - No tracks have `key == -1`, so we don't need to handle "no key found".

## 3. Data cleaning

We clean the data step by step. After each step we print how many rows were removed and why.
We never change the raw file. We work on a copy called `df`.

In [109]:
df = raw.copy()

### 3.1 Drop the index column

`Unnamed: 0` is just the row number from when the file was saved. It tells us nothing about the song.

In [110]:
before = df
df = df.drop(columns=["Unnamed: 0"])
log_step("Drop index column", before, df, "Row number only, not a feature")

Drop index column: 114,000 -> 114,000 rows (0 removed)


### 3.2 Missing values

Only 1 row has missing values. We drop it. 

In [111]:
display(df[df.isna().any(axis=1)])

before = df
df = df.dropna()
log_step("Drop missing values", before, df, "No artist, album, or track name (and duration 0)")

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
65900,1kR4gIb7nGxHPI3D2ifs59,NaN,NaN,NaN,0,0,False,0.501,0.583,7,-9.46,0,0.0605,0.69,0.00396,0.0747,0.734,138.391,4,k-pop


Drop missing values: 114,000 -> 113,999 rows (1 removed)


### 3.3 Duplicate songs: same `track_id`

The same song can show up under more than one genre with the same `track_id`.
If we keep all copies, the same song could end up in both the train and test sets. That is data leakage.

Here is the song that appears the most times. Only the genre changes. The audio features and valence are the same in every row.

In [112]:
top_id = df["track_id"].value_counts().index[0]
df[df["track_id"] == top_id][["track_name", "artists", "track_genre", "danceability", "energy", "valence"]]

,track_name,artists,track_genre,danceability,energy,valence
8315,Baby Blue - Remastered 2010,Badfinger,blues,0.623,0.876,0.515
19759,Baby Blue - Remastered 2010,Badfinger,country,0.623,0.876,0.515
34728,Baby Blue - Remastered 2010,Badfinger,folk,0.623,0.876,0.515
62226,Baby Blue - Remastered 2010,Badfinger,j-pop,0.623,0.876,0.515
63087,Baby Blue - Remastered 2010,Badfinger,j-rock,0.623,0.876,0.515
82064,Baby Blue - Remastered 2010,Badfinger,power-pop,0.623,0.876,0.515
84129,Baby Blue - Remastered 2010,Badfinger,psych-rock,0.623,0.876,0.515
99727,Baby Blue - Remastered 2010,Badfinger,singer-songwriter,0.623,0.876,0.515
102732,Baby Blue - Remastered 2010,Badfinger,songwriter,0.623,0.876,0.515


**Which genre do we keep?** When a song has several genres, we keep only one row, so we have to pick one genre.

The raw file is sorted by genre in alphabetical order (acoustic, afrobeat, alt-rock, ...).
If we simply kept the first row, every song would get the genre that comes first in the alphabet. In the example above, that is always "blues".
Genres near the start of the alphabet would keep many songs, and genres near the end would lose many.

So we **pick one of the song's genres at random** (with `random_state=42`, so the result is the same every time).
This way no genre is favored just because of its name.

In [113]:
before = df
df = (
    df.sample(frac=1, random_state=RANDOM_STATE)   # shuffle the rows
      .drop_duplicates(subset="track_id", keep="first")   # keep one random genre per song
      .sort_index()                                # put the rows back in the original order
)
log_step("Drop duplicate track_id", before, df, "Same song listed under several genres")

Drop duplicate track_id: 113,999 -> 89,740 rows (24,259 removed)


### 3.4 Duplicate songs: same name and artist

The same song can also have **different** `track_ids`, for example when it is on both a single and an album, or on a "Greatest Hits" album.
We treat songs with the same `track_name` and `artists` as one song.

Before we drop them, we check that these really are the same song. If they are, their audio features should be almost the same.

In [114]:
same_name = df[df.duplicated(["track_name", "artists"], keep=False)].sort_values(["artists", "track_name"])
display(same_name[["track_name", "artists", "album_name", "track_genre", "energy", "valence", "duration_ms"]].head(10))

# For each (name, artist) group, how much does valence change between copies?
valence_range = same_name.groupby(["track_name", "artists"])["valence"].agg(lambda s: s.max() - s.min())
print(f"Groups of same name + artist: {len(valence_range):,}")
print(f"Groups where valence differs by less than 0.1: {(valence_range < 0.1).mean():.1%}")

,track_name,artists,album_name,track_genre,energy,valence,duration_ms
58516,Crazy Legs,(Hed) P.E.,Broke,industrial,0.775,0.736,242800
58857,Crazy Legs,(Hed) P.E.,Broke,industrial,0.810,0.738,242960
58306,Waiting to Die,(Hed) P.E.,Broke,industrial,0.962,0.484,194800
58874,Waiting to Die,(Hed) P.E.,Broke,industrial,0.964,0.533,194626
46055,Dreams (Will Come Alive),2 Brothers On The 4th Floor,Dreams,happy,0.953,0.507,264000
46065,Dreams (Will Come Alive),2 Brothers On The 4th Floor,Dreams (Will Come Alive),happy,0.979,0.612,263162
46484,Fairytales - Charly Lownoise & Mental Theo Rav...,2 Brothers On The 4th Floor,The Very Best of 2 Brothers on the 4th Floor (...,happy,0.964,0.571,246969
46997,Fairytales - Charly Lownoise & Mental Theo Rav...,2 Brothers On The 4th Floor,2,happy,0.964,0.571,246969
46209,Fly,2 Brothers On The 4th Floor,Fly (Through The Starry Night),happy,0.898,0.759,208227
46709,Fly,2 Brothers On The 4th Floor,The Very Best of 2 Brothers on the 4th Floor (...,happy,0.898,0.754,208227


Groups of same name + artist: 4,657
Groups where valence differs by less than 0.1: 96.1%


The copies are on different albums, but their features are almost the same, and in about 96% of the groups valence differs by less than 0.1.
So they are the same recording released more than once. We keep one row for each.

In [115]:
before = df
df = df.drop_duplicates(subset=["track_name", "artists"], keep="first")
log_step("Drop duplicate name + artist", before, df, "Same song re-released with a new track_id")

Drop duplicate name + artist: 89,740 -> 81,343 rows (8,397 removed)


#### Same audio, different title

After the split (step 3.9), we found that a few songs in the test set were exactly the same as songs in the training set in every audio feature.
They have different titles, so the name + artist check above missed them. Here are some:

In [116]:
audio_cols = ["duration_ms", "danceability", "energy", "key", "loudness", "mode", "speechiness",
              "acousticness", "instrumentalness", "liveness", "valence", "tempo"]

same_audio = df[df.duplicated(audio_cols, keep=False)].sort_values(audio_cols)
display(same_audio[["track_name", "artists", "track_genre", "duration_ms", "valence"]].head(10))
print(f"Extra rows with identical audio features: {df.duplicated(audio_cols).sum():,}")

,track_name,artists,track_genre,duration_ms,valence
26974,Jasmin begegnet Prinz Anders,Alan Menken,disney,34878,0.1070
98804,Jasmine Meets Prince Anders,Alan Menken,show-tunes,34878,0.1070
16046,"Vom Himmel hoch da komm ich her, BWV 769 - Orc...",Johann Sebastian Bach;Boston Symphony Orchestr...,classical,47000,0.2550
16537,"Vom Himmel hoch da komm ich her, BWV 769 - Orc...",Boston Symphony Orchestra;Seiji Ozawa,classical,47000,0.2550
26240,La tour de Bruno,Germaine Franco,disney,52466,0.0329
26376,Brunos Turm,Germaine Franco,disney,52466,0.0329
26797,Bruno's toren,Germaine Franco,disney,52466,0.0329
26341,Bruno's Tower,Germaine Franco,disney,52466,0.0343
26596,Wieża Bruna,Germaine Franco,disney,52466,0.0343
101483,Pure Brown Noise with Pouring Rain,White Noise Therapy;Rain Sounds;Natural White ...,sleep,61020,0.0000


Extra rows with identical audio features: 406


These are the same recording with a translated title or the same song with a slightly different artist name. Twelve audio features matching to many decimal places by chance is very unlikely, so we treat them as the same song and keep one.

In [117]:
before = df
df = df.drop_duplicates(subset=audio_cols, keep="first")
log_step("Drop identical audio features", before, df, "Same recording with a translated or different title")

Drop identical audio features: 81,343 -> 80,937 rows (406 removed)


### 3.5 Tracks that are not really music

We want to predict how sad a **song** sounds. Some tracks are not songs, and their valence does not mean the same thing.
We look at three groups: tracks with very high `speechiness`, tracks with `tempo == 0`, and the `comedy` and `sleep` genres.

#### Very high speechiness

Spotify says that tracks with `speechiness` above 0.66 are probably made almost entirely of spoken words (audiobooks, podcasts, stand-up comedy).

In [118]:
talky = df[df["speechiness"] > 0.66]
print(f"Tracks with speechiness > 0.66: {len(talky):,}")
print(talky["track_genre"].value_counts().head(5))

Tracks with speechiness > 0.66: 859
track_genre
comedy        764
kids           11
children       10
funk            9
show-tunes      9
Name: count, dtype: int64


Most of these tracks (764 of 859) are from the `comedy` genre, which is mostly stand-up comedy. We remove all tracks above 0.66.

In [119]:
before = df
df = df[df["speechiness"] <= 0.66]
log_step("Drop speechiness > 0.66", before, df, "Mostly spoken words, not music")

Drop speechiness > 0.66: 80,937 -> 80,078 rows (859 removed)


#### Tempo of 0

A real song always has some tempo. Let's look at the tracks with `tempo == 0`.

In [120]:
no_beat = df[df["tempo"] == 0]
print(f"Tracks with tempo == 0: {len(no_beat):,}")
print(no_beat["track_genre"].value_counts().head(5), "\n")
no_beat[["danceability", "speechiness", "valence", "time_signature"]].describe().loc[["mean", "min", "max"]]

Tracks with tempo == 0: 124
track_genre
sleep          105
guitar           4
iranian          4
ambient          3
world-music      3
Name: count, dtype: int64 



,danceability,speechiness,valence,time_signature
mean,0.0,0.0,0.0,0.0
min,0.0,0.0,0.0,0.0
max,0.0,0.0,0.0,0.0


For every one of these tracks, `danceability`, `speechiness`, `time_signature`, and even **`valence` are exactly 0**.
This means Spotify could not analyze them at all. Most are from the `sleep` genre (white noise, rain sounds).
A valence of 0 here does not mean "very sad", it means "no result". Keeping them would teach the model wrong answers, so we remove them.

In [121]:
before = df
df = df[df["tempo"] != 0]
log_step("Drop tempo == 0", before, df, "Spotify could not analyze these tracks (all features 0)")

Drop tempo == 0: 80,078 -> 79,954 rows (124 removed)


#### The `comedy` and `sleep` genres

Even after the two steps above, some tracks from these two genres are left. We look at a few of them.

In [122]:
for genre in ["comedy", "sleep"]:
    g = df[df["track_genre"] == genre]
    print(f"--- {genre}: {len(g)} tracks left, median speechiness {g['speechiness'].median():.2f}, median valence {g['valence'].median():.2f}")
    display(g[["track_name", "artists", "speechiness", "valence"]].sample(8, random_state=RANDOM_STATE))

--- comedy: 227 tracks left, median speechiness 0.17, median valence 0.64


,track_name,artists,speechiness,valence
18009,Professional Rapper (feat. Snoop Dogg),Lil Dicky;Snoop Dogg,0.3620,0.537
18587,Happily Ever After,Kevin Bloody Wilson,0.1660,0.876
18015,Crazy Rap (Colt 45 & 2 Zig Zags),Afroman,0.3820,0.576
18498,Civil War Again,Kathleen Madigan,0.5870,0.540
18612,Bed Pan Man,Tim Wilson,0.4780,0.965
18250,420,Afroman;Yung Fate;DJLeach;Jake Strain,0.0975,0.783
18501,Show Me Your Genitals 2: E=MC Vagina,Jon Lajoie,0.2240,0.855
18363,Professor Whiskers,Aunty Donna,0.2670,0.495


--- sleep: 804 tracks left, median speechiness 0.05, median valence 0.04


,track_name,artists,speechiness,valence
101234,Den bortkomne sauen (Arr. G. Eriksson for 2 So...,Traditional;The Norwegian Soloists' Choir;Gret...,0.0457,0.21300
101889,Calm Baby Lullabies,Baby Sleep Music;Baby Lullaby;Monarch Baby Lul...,0.0394,0.04370
101202,Hush,Yonatan Awate,0.0438,0.06890
101650,Endless Sky,Nils August Valentin,0.0377,0.18100
101665,Shhhh Waves and Gulls,Silent Knights,0.0446,0.00001
101978,Bultande Hjärta,Avslappning Sound,0.0479,0.12800
101864,Rain at Breakfast,Outside Broadcast Recordings,0.0875,0.02120
101403,Monrovia,Prescient Liquids,0.0445,0.04070


**Decision:** we remove both genres.

- **`comedy`:** the tracks left are a mix of comedy songs and stand-up bits with a lot of talking. Their mood comes from the jokes (the words), not from how they sound. Our features only describe sound.
- **`sleep`:** most tracks are white noise, rain, or ocean sounds, not music. Their valence is almost always close to 0. If we kept them, the model would learn that "quiet noise = very sad", which is not what we mean by a sad song.

We checked the other genres too. No other genre has a high median speechiness (the next highest is `j-dance` at about 0.2), so they are kept.

In [123]:
before = df
df = df[~df["track_genre"].isin(["comedy", "sleep"])]
log_step("Drop comedy and sleep genres", before, df, "Stand-up comedy and noise/nature sounds, not music")

Drop comedy and sleep genres: 79,954 -> 78,923 rows (1,031 removed)


### 3.6 Odd values in `time_signature` and `key`

We check these two columns again after the cleaning above.

In [124]:
print(df["time_signature"].value_counts().sort_index(), "\n")
print("Tracks with key == -1:", (df["key"] == -1).sum())

time_signature
0        5
1      656
3     6307
4    70691
5     1264
Name: count, dtype: int64 

Tracks with key == -1: 0


- Most of the `time_signature == 0` tracks were the ones with `tempo == 0`, so only a few are left.
- No tracks have `key == -1`, so there is nothing to fix.

### 3.7 Drop columns that are not features

`track_id`, `artists`, `album_name`, and `track_name` are names and IDs, not things you can hear in a song.
We drop them **after** removing duplicates, since we needed them for that.

**Why not use `artists` as a feature?** Our question is about how a song sounds, not who made it. Songs by the same artist often have similar valence, so a model could just memorize artists instead of learning from the sound. There are also about 26,000 different artists, and most have only one song, so the model could not learn much from them anyway.

One limit to keep in mind: the same artist can still have songs in both the training and test sets. Because their songs sound alike, this may make test scores a little better than they would be for a completely new artist.

 We also drop `popularity`. Our question is whether we can tell how sad a song is from **how it sounds**, and popularity is not a sound. 

In [125]:
before = df
df = df.drop(columns=["track_id", "artists", "album_name", "track_name", "popularity"]).reset_index(drop=True)
log_step("Drop name/ID columns and popularity", before, df, "Not audio features (rows unchanged)")

print("Columns left:", list(df.columns))

Drop name/ID columns and popularity: 78,923 -> 78,923 rows (0 removed)
Columns left: ['duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']


### 3.8 Cleaning summary

In [126]:
summary_table = pd.DataFrame(cleaning_log)
summary_table

,step,rows_before,rows_after,rows_removed,reason
0,Drop index column,114000,114000,0,"Row number only, not a feature"
1,Drop missing values,114000,113999,1,"No artist, album, or track name (and duration 0)"
2,Drop duplicate track_id,113999,89740,24259,Same song listed under several genres
3,Drop duplicate name + artist,89740,81343,8397,Same song re-released with a new track_id
4,Drop identical audio features,81343,80937,406,Same recording with a translated or different ...
5,Drop speechiness > 0.66,80937,80078,859,"Mostly spoken words, not music"
6,Drop tempo == 0,80078,79954,124,Spotify could not analyze these tracks (all fe...
7,Drop comedy and sleep genres,79954,78923,1031,"Stand-up comedy and noise/nature sounds, not m..."
8,Drop name/ID columns and popularity,78923,78923,0,Not audio features (rows unchanged)


In [127]:
# Genres are no longer exactly 1,000 songs each. How uneven are they now?
genre_counts = df["track_genre"].value_counts()
print(f"Genres left: {df['track_genre'].nunique()}")
print(f"Songs per genre: min {genre_counts.min()} ({genre_counts.idxmin()}), max {genre_counts.max()} ({genre_counts.idxmax()})")

Genres left: 112
Songs per genre: min 210 (reggaeton), max 994 (study)


We started with 114,000 rows and ended with 78,923 songs. Almost all of the removed rows were **duplicate songs** (about 33,000).
Removing tracks that are not music took away only about 2,000 rows.

Because duplicates were spread unevenly, the genres no longer have exactly 1,000 songs each. They now range from 210 songs (`reggaeton`) to 994 (`study`).
Broad genres that share many songs with other genres lost the most (for example `reggaeton`, `reggae`, `rock`, `songwriter`, `alternative`).
Genres with their own distinct sound, like `heavy-metal`, `grindcore`, and `bluegrass`, kept almost all of their songs.
The team should keep this in mind when looking at results by genre.

### 3.9 Shared train/test split

We now split the cleaned data into a **training set (80%)** and a **test set (20%)**.

- **Training set:** used for everything before the final score: data exploration, feature engineering, training models, and picking settings with cross-validation.
- **Test set:** kept aside and used **only once at the very end**, to give each final model an honest score on songs it has never seen.

**Stratify on the "sad" label:** for the classification task, a song is "sad" if `valence < 0.3`.
We stratify the split on this label, so the training and test sets have the same share of sad songs.

We use `random_state=42`, so everyone in the team gets exactly the same split.

In [128]:
is_sad = (df["valence"] < 0.3).astype(int)   # only used to stratify the split here
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=RANDOM_STATE, stratify=is_sad
)

print(f"Training set: {len(train_df):,} songs")
print(f"Test set:     {len(test_df):,} songs")
print(f"Share of sad songs (valence < 0.3): train {(train_df['valence'] < 0.3).mean():.1%}, test {(test_df['valence'] < 0.3).mean():.1%}")

Training set: 63,138 songs
Test set:     15,785 songs
Share of sad songs (valence < 0.3): train 31.0%, test 31.0%


We save both sets as CSV files in the project folder, next to `raw_dataset.csv`. The files contain the **cleaned but not yet encoded or scaled** data, so teammates can explore the original features.
Encoding and scaling come in Part 4, and are fit on the training set only.

In [129]:
train_df.to_csv("train_clean.csv", index=False)
test_df.to_csv("test_clean.csv", index=False)
print("Saved: train_clean.csv and test_clean.csv")

Saved: train_clean.csv and test_clean.csv


**For teammates:** load the shared split like this.

```python
train_df = pd.read_csv("train_clean.csv")
test_df = pd.read_csv("test_clean.csv")
```

- `valence` is the target. Never use it as an input.
- Do all exploration and tuning on `train_df`. Use `test_df` only for the final score.